# Chapter 10. Building Neural Networks with PyTorch

PyTorch is a deep learning library for building and training neural networks.

Like NumPy, it provides multidimensional arrays for numerical computation. In addition, it supports automatic differentiation and hardware acceleration.

## PyTorch Fundamentals

The core data structure in PyTorch is the tensor.

A tensor is a multidimensional array with a shape and data type, similar to a NumPy array. Unlike NumPy arrays, tensors can use hardware accelerators and support automatic differentiation.

### PyTorch Tensors

In [6]:
import torch

X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

print(X)

tensor([[1., 4., 7.],
        [2., 3., 6.]])


Like NumPy arrays, tensors have a shape and a data type.

In [13]:
print("Shape:", X.shape)
print("\nData type:", X.dtype)

Shape: torch.Size([2, 3])

Data type: torch.float32


Tensor indexing and slicing work similarly to NumPy.

In [22]:
print(X[0, 1])
print(X[:, 1])

tensor(4.)
tensor([4., 3.])


PyTorch supports elementwise operations, reductions, and matrix multiplication.

In [27]:
10 * (X + 1.0) # itemwise addition and multiplication

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [29]:
X.exp() # itemwise exponential

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [31]:
X.mean()

tensor(3.8333)

In [35]:
X.max(dim=0) # max values along dimension 0 (i.e., max value per column)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [37]:
X @ X.T # matrix transpose and matrix multiplication

tensor([[66., 56.],
        [56., 49.]])

Tensors can be converted to NumPy arrays, and NumPy arrays can be converted to tensors.

PyTorch uses `float32` by default for floating-point tensors, while NumPy usually uses `float64`.

In [54]:
import numpy as np

print("Tensor converted to a NumPy array:")
X.numpy()

Tensor converted to a NumPy array:


array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [58]:
print("Tensor created from a NumPy array:")
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

Tensor created from a NumPy array:


tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

Using `float32` is common in deep learning because it uses less memory and is generally faster than `float64`, while providing enough precision for most neural network tasks.

PyTorch supports in-place operations, which modify an existing tensor directly.

These operations end with an underscore, such as `relu_()` or `zero_()`.

In [68]:
X[:, 1] = -99

print("After modifying the second column:")
X

After modifying the second column:


tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [70]:
X.relu_()

print("After applying ReLU in place:")
X

After applying ReLU in place:


tensor([[1., 0., 7.],
        [2., 0., 6.]])

In-place operations can save memory, but they should be used carefully when working with autograd because modifying tensors needed for backpropagation can cause errors.

### Hardware Acceleration

PyTorch tensors can run on GPUs and other supported accelerators.

Using a GPU can greatly speed up large matrix operations and neural network training.

The following code checks whether an NVIDIA GPU is available through CUDA. If not, it checks for Apple's Metal Performance Shaders (MPS). Otherwise, it uses the CPU.

In [77]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Selected device:", device)

Selected device: cuda


A tensor can be moved to the selected device using `.to(device)`.

Once a tensor is on a GPU, operations involving that tensor will also run on the GPU.

In [85]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

M = M.to(device)

print("Tensor device:")
print(M.device)

Tensor device:
cuda:0


Alternatively, a tensor can be created directly on the selected device.

In [89]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

print("Tensor device:")
print(M.device)

Tensor device:
cuda:0


In [93]:
R = M @ M.T # run some operations on the GPU

print("Matrix multiplication result:")
print(R)

print("\nResult device:")
print(R.device)

Matrix multiplication result:
tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

Result device:
cuda:0


Keeping tensors and model parameters on the same device avoids repeated CPU-to-GPU transfers.

Data transfer can become a bottleneck, especially when the model is small or the batches are too small.

GPUs are most effective for large parallel computations.

For small operations, the overhead of moving data and coordinating GPU work can reduce or eliminate the speed advantage.

In [97]:
M = torch.rand((1000, 1000)) # on the CPU

%timeit M @ M.T

13.2 ms ± 165 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [99]:
M = torch.rand((1000, 1000), device="cuda") # on the GPU

%timeit M @ M.T

126 μs ± 1.47 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Autograd

PyTorch's autograd automatically tracks tensor operations and computes gradients during backpropagation.

To track gradients, create tensors with `requires_grad=True`.

In [118]:
x = torch.tensor(5.0, requires_grad=True)

f = x ** 2

print("f:")
print(f)

f:
tensor(25., grad_fn=<PowBackward0>)


In [120]:
f.backward()

print("Gradient of f with respect to x:")
print(x.grad)

Gradient of f with respect to x:
tensor(10.)


For $f(x) = x^2$, autograd computes:

$$
\frac{df}{dx} = 2x
$$

At $x = 5$, the gradient is $10$.

`backward()` performs the backward pass and stores gradients in the `.grad` attribute of tracked tensors.

In [124]:
learning_rate = 0.1

with torch.no_grad():
    x -= learning_rate * x.grad # gradient descent step

print("Updated x:")
print(x)

Updated x:
tensor(4., requires_grad=True)


Use `torch.no_grad()` when updating parameters or making predictions.

This prevents PyTorch from tracking these operations in the computation graph.

In [127]:
x.grad.zero_()

print("Gradient after reset:")
print(x.grad)

Gradient after reset:
tensor(0.)


PyTorch accumulates gradients by default.

Therefore, gradients should be reset before the next backward pass.

In [132]:
learning_rate = 0.1

x = torch.tensor(5.0, requires_grad=True)

for iteration in range(100):
    f = x ** 2 # forward pass
    f.backward() # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_() # reset the gradients

print("Final x:")
print(x)

Final x:
tensor(1.0185e-09, requires_grad=True)


Each iteration follows this pattern:

1. Forward pass: compute the output and loss
2. Backward pass: compute gradients with `backward()`
3. Update parameters
4. Reset gradients

The `zero_()` method is an in-place operation, meaning that it modifies the existing gradient tensor directly.

In-place operations end with `_` and can save memory, but they should be used carefully because modifying a tensor needed for backpropagation can cause a `RuntimeError`.

## Implementing Linear Regression

We will implement linear regression directly with PyTorch tensors and autograd.

The model predicts:

$$
\hat{y} = Xw + b
$$

### Linear Regression Using Tensors and Autograd

We use the California housing dataset and split it into training, validation, and test sets.

In [160]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data,
    housing.target,
    test_size=0.2,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_valid.shape)
print("Test set shape:", X_test.shape)

Training set shape: (13209, 8)
Validation set shape: (3303, 8)
Test set shape: (4128, 8)


Convert the inputs and targets to `float32` tensors.

The input features are standardized using the training-set mean and standard deviation.

In [163]:
X_train = torch.from_numpy(X_train).float()
X_valid = torch.from_numpy(X_valid).float()
X_test = torch.from_numpy(X_test).float()

means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)

X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [165]:
y_train = torch.from_numpy(y_train).float().reshape(-1, 1)
y_valid = torch.from_numpy(y_valid).float().reshape(-1, 1)
y_test = torch.from_numpy(y_test).float().reshape(-1, 1)

Create one weight per input feature and one bias term.

`requires_grad=True` tells autograd to compute gradients for these parameters.

In [170]:
torch.manual_seed(42)

n_features = X_train.shape[1] # there are 8 input features

w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

We use batch gradient descent, so each update uses the full training set.

In [179]:
learning_rate = 0.4
n_epochs = 20

for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item():.4f}")

Epoch 1/20, Loss: 0.5521
Epoch 2/20, Loss: 0.5485
Epoch 3/20, Loss: 0.5453
Epoch 4/20, Loss: 0.5423
Epoch 5/20, Loss: 0.5397
Epoch 6/20, Loss: 0.5373
Epoch 7/20, Loss: 0.5352
Epoch 8/20, Loss: 0.5333
Epoch 9/20, Loss: 0.5315
Epoch 10/20, Loss: 0.5299
Epoch 11/20, Loss: 0.5285
Epoch 12/20, Loss: 0.5272
Epoch 13/20, Loss: 0.5260
Epoch 14/20, Loss: 0.5249
Epoch 15/20, Loss: 0.5239
Epoch 16/20, Loss: 0.5230
Epoch 17/20, Loss: 0.5222
Epoch 18/20, Loss: 0.5214
Epoch 19/20, Loss: 0.5207
Epoch 20/20, Loss: 0.5201


Use `torch.no_grad()` during inference because gradients are not needed.

In [184]:
X_new = X_test[:3] # pretend these are new instances

with torch.no_grad():
    y_pred = X_new @ w + b # use the trained parameters to make predictions

print("Predictions:")
print(y_pred)

Predictions:
tensor([[0.7924],
        [1.7043],
        [2.7219]])


### Linear Regression Using PyTorch's High-Level API

PyTorch provides higher-level modules for defining models, loss functions, and parameter updates.

This reduces the amount of manual tensor and gradient-management code.

In [187]:
import torch.nn as nn

torch.manual_seed(42)

model = nn.Linear(in_features=n_features, out_features=1)

`nn.Linear` creates a linear layer that computes:

$$
XW^\top + b
$$

The module automatically creates trainable weights and biases.

In [192]:
print("Bias:")
print(model.bias)

print("\nWeights:")
print(model.weight)

Bias:
Parameter containing:
tensor([0.3117], requires_grad=True)

Weights:
Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)


A PyTorch module can be called like a function.

Calling `model(X)` runs the module's `forward()` method.

In [197]:
model(X_train[:2])

tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

Use an optimizer to update the model parameters and a criterion to compute the loss.

In [200]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

The training loop is similar to the earlier version, but the optimizer now handles parameter updates and gradient resets.

In [203]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = mse(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch{epoch + 1}/{n_epochs}, Loss: {loss.item():.4f}")

In [205]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch1/20, Loss: 4.2702
Epoch2/20, Loss: 0.7621
Epoch3/20, Loss: 0.6113
Epoch4/20, Loss: 0.5920
Epoch5/20, Loss: 0.5812
Epoch6/20, Loss: 0.5728
Epoch7/20, Loss: 0.5657
Epoch8/20, Loss: 0.5596
Epoch9/20, Loss: 0.5543
Epoch10/20, Loss: 0.5496
Epoch11/20, Loss: 0.5456
Epoch12/20, Loss: 0.5420
Epoch13/20, Loss: 0.5389
Epoch14/20, Loss: 0.5361
Epoch15/20, Loss: 0.5337
Epoch16/20, Loss: 0.5316
Epoch17/20, Loss: 0.5297
Epoch18/20, Loss: 0.5280
Epoch19/20, Loss: 0.5265
Epoch20/20, Loss: 0.5252


After training, use the model inside `torch.no_grad()` for inference.

In [210]:
X_new = X_test[:3] # pretend these are new instances

with torch.no_grad():
    y_pred = model(X_new)

print("Predictions:")
print(y_pred)

Predictions:
tensor([[0.8241],
        [1.6844],
        [2.6655]])


## Implementing a Regression MLP

We can build a multilayer perceptron by stacking linear layers and activation functions.

This model has two hidden layers and one output layer.

In [217]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

The model structure is:

- `n_features` → 50 neurons
- ReLU activation
- 50 → 40 neurons
- ReLU activation
- 40 → 1 output neuron

The output layer has no activation function because this is a regression task.

Each `nn.Linear` layer applies a linear transformation.

The `nn.ReLU()` layers add nonlinearity, allowing the model to learn more complex relationships than linear regression.

In [223]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch1/20, Loss: 4.9719
Epoch2/20, Loss: 2.0490
Epoch3/20, Loss: 1.0056
Epoch4/20, Loss: 0.8662
Epoch5/20, Loss: 0.7839
Epoch6/20, Loss: 0.7322
Epoch7/20, Loss: 0.6987
Epoch8/20, Loss: 0.6764
Epoch9/20, Loss: 0.6608
Epoch10/20, Loss: 0.6492
Epoch11/20, Loss: 0.6399
Epoch12/20, Loss: 0.6319
Epoch13/20, Loss: 0.6249
Epoch14/20, Loss: 0.6183
Epoch15/20, Loss: 0.6122
Epoch16/20, Loss: 0.6064
Epoch17/20, Loss: 0.6008
Epoch18/20, Loss: 0.5954
Epoch19/20, Loss: 0.5901
Epoch20/20, Loss: 0.5851


## Implementing Mini-Batch Gradient Descent Using DataLoaders

`DataLoader` loads data in mini-batches and can shuffle the training set at each epoch.

This is more scalable than batch gradient descent and works well with GPUs.

In [234]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Number of batches:", len(train_loader))

Number of batches: 413


`TensorDataset` stores the input features and targets together.

At each iteration, `train_loader` returns one batch of `X_batch` and `y_batch`.

Move the model to the selected device before creating the optimizer.

This ensures that the model parameters and optimizer state are stored on the same device.

In [243]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

model = model.to(device)

learning_rate = 0.02

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

mse = nn.MSELoss()

The training loop processes one mini-batch at a time.

`model.train()` switches the model to training mode.

In [251]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    
    for epoch in range(n_epochs):
        total_loss = 0.
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            # Forward pass
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            # Track batch loss
            total_loss += loss.item()
            # Backward pass and parameter update
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [253]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.5894
Epoch 2/20, Loss: 0.4202
Epoch 3/20, Loss: 0.3808
Epoch 4/20, Loss: 0.3656
Epoch 5/20, Loss: 0.3560
Epoch 6/20, Loss: 0.3499
Epoch 7/20, Loss: 0.3445
Epoch 8/20, Loss: 0.3408
Epoch 9/20, Loss: 0.3367
Epoch 10/20, Loss: 0.3324
Epoch 11/20, Loss: 0.3282
Epoch 12/20, Loss: 0.3241
Epoch 13/20, Loss: 0.3226
Epoch 14/20, Loss: 0.3186
Epoch 15/20, Loss: 0.3128
Epoch 16/20, Loss: 0.3142
Epoch 17/20, Loss: 0.3093
Epoch 18/20, Loss: 0.3068
Epoch 19/20, Loss: 0.3063
Epoch 20/20, Loss: 0.3016


After training, switch the model to evaluation mode before making predictions.

`model.eval()` is important for layers that behave differently during training and evaluation, such as dropout and batch normalization.

## Model Evaluation

Use `model.eval()` and `torch.no_grad()` when evaluating a trained model.

In [259]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
            
    return aggregate_fn(torch.stack(metrics))

`metric_fn` computes a metric for each batch.

`aggregate_fn` combines the batch-level metrics into one final value.

In [264]:
valid_dataset = TensorDataset(X_valid, y_valid)

valid_loader = DataLoader(valid_dataset, batch_size=32)

valid_mse = evaluate(model, valid_loader, mse)

print("Validation MSE:")
print(valid_mse)

Validation MSE:
tensor(0.3242, device='cuda:0')


RMSE is often easier to interpret than MSE because it uses the same unit as the target.

However, averaging batch-level RMSE values is not equivalent to computing RMSE over the full validation set.

In [267]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

In [271]:
batch_rmse = evaluate(model, valid_loader, rmse)

print("Mean batch RMSE:")
print(batch_rmse)

print("\nSquare root of validation MSE:")
print(valid_mse.sqrt())

Mean batch RMSE:
tensor(0.5549, device='cuda:0')

Square root of validation MSE:
tensor(0.5694, device='cuda:0')


To compute the correct overall RMSE, first aggregate MSE across batches and then take the square root.

In [274]:
valid_rmse = evaluate(model, valid_loader, mse,
         aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

print("Validation RMSE:")
print(valid_rmse)

Validation RMSE:
tensor(0.5694, device='cuda:0')


## Building Nonsequential Models Using Custom Modules

`nn.Sequential` works well for a simple stack of layers.

For models with branches, skip connections, or multiple paths, define a custom class that inherits from `nn.Module`.

A Wide & Deep model combines:

- a deep path that learns complex patterns
- a wide path that passes the original input directly to the output layer

In [288]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

The `forward()` method defines how data moves through the model.

The original input `X` is concatenated with the output of the deep stack before the final prediction layer.

The deep stack outputs 40 features.

Since the original input contains `n_features` columns, the final layer receives `40 + n_features` inputs.

In [292]:
torch.manual_seed(42)

model = WideAndDeep(n_features).to(device)

learning_rate = 0.002

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

mse = nn.MSELoss()

print(model)

WideAndDeep(
  (deep_stack): Sequential(
    (0): Linear(in_features=8, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=40, bias=True)
    (3): ReLU()
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)


Custom modules can be trained, evaluated, and used exactly like built-in PyTorch modules.

In [295]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 1.3319
Epoch 2/20, Loss: 0.6117
Epoch 3/20, Loss: 0.5662
Epoch 4/20, Loss: 0.5345
Epoch 5/20, Loss: 0.5104
Epoch 6/20, Loss: 0.4929
Epoch 7/20, Loss: 0.4804
Epoch 8/20, Loss: 0.4683
Epoch 9/20, Loss: 0.4589
Epoch 10/20, Loss: 0.4501
Epoch 11/20, Loss: 0.4423
Epoch 12/20, Loss: 0.4354
Epoch 13/20, Loss: 0.4282
Epoch 14/20, Loss: 0.4228
Epoch 15/20, Loss: 0.4167
Epoch 16/20, Loss: 0.4116
Epoch 17/20, Loss: 0.4075
Epoch 18/20, Loss: 0.4018
Epoch 19/20, Loss: 0.3971
Epoch 20/20, Loss: 0.3934


### Building Models with Multiple Inputs

Some models need separate inputs instead of one combined tensor.

For example, a model may process tabular features, images, or text through different paths before combining their outputs.

In this Wide & Deep model, the wide path uses the first 5 features, while the deep path uses features starting from index 2.

The two feature groups overlap because features at indices 2, 3, and 4 are used in both paths.

In [308]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_wide_features, n_deep_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_deep_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU()
        )
        self.output_layer = nn.Linear(n_wide_features + 40, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

The `forward()` method now receives two tensors:

- `X_wide` goes directly to the final layer.
- `X_deep` passes through the deep stack first.

Their outputs are concatenated before the final prediction layer.

In [311]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)

valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
valid_loader_wd = DataLoader(valid_data_wd, batch_size=32)

test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32)

Each batch now contains three tensors:

1. wide input features
2. deep input features
3. target values

The training and evaluation loops must be updated because the model now receives two input tensors.

In [319]:
def train_multiple_inputs(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch_wide, X_batch_deep, y_batch in train_loader:
            X_batch_wide = X_batch_wide.to(device)
            X_batch_deep = X_batch_deep.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch_wide, X_batch_deep)

            loss = criterion(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)

        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [323]:
torch.manual_seed(42)

model = WideAndDeepV3(n_wide_features=5, n_deep_features=n_features-2).to(device)

learning_rate = 0.002

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

mse = nn.MSELoss()

train_multiple_inputs(model, optimizer, mse, train_loader_wd, n_epochs)

Epoch 1/20, Loss: 1.3836
Epoch 2/20, Loss: 0.6061
Epoch 3/20, Loss: 0.5504
Epoch 4/20, Loss: 0.5176
Epoch 5/20, Loss: 0.4971
Epoch 6/20, Loss: 0.4822
Epoch 7/20, Loss: 0.4706
Epoch 8/20, Loss: 0.4619
Epoch 9/20, Loss: 0.4547
Epoch 10/20, Loss: 0.4486
Epoch 11/20, Loss: 0.4426
Epoch 12/20, Loss: 0.4383
Epoch 13/20, Loss: 0.4341
Epoch 14/20, Loss: 0.4307
Epoch 15/20, Loss: 0.4277
Epoch 16/20, Loss: 0.4244
Epoch 17/20, Loss: 0.4227
Epoch 18/20, Loss: 0.4199
Epoch 19/20, Loss: 0.4176
Epoch 20/20, Loss: 0.4159


Instead of naming each input tensor separately, starred unpacking can collect all model inputs into `X_batch_inputs`.

The target tensor is kept separate as `y_batch`.

In [326]:
def train_multiple_inputs(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for *X_batch_inputs, y_batch in train_loader:
            X_batch_inputs = [X_batch.to(device) for X_batch in X_batch_inputs]
            y_batch = y_batch.to(device)

            y_pred = model(*X_batch_inputs)

            loss = criterion(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)

        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

For this dataset, `X_batch_inputs` contains two tensors:

1. wide input features
2. deep input features

Therefore, `model(*X_batch_inputs)` is equivalent to:

`model(X_batch_wide, X_batch_deep)`

When a model has many inputs, relying on input order can cause mistakes.

A custom dataset can return a dictionary that maps each input name to its tensor.

In [340]:
class WideAndDeepDataset(torch.utils.data.Dataset):
    def __init__(self, X_wide, X_deep, y):
        self.X_wide = X_wide
        self.X_deep = X_deep
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        input_dict = {"X_wide": self.X_wide[idx], "X_deep": self.X_deep[idx]}
        return input_dict, self.y[idx]

The dictionary keys match the parameter names in:

`forward(self, X_wide, X_deep)`

In [345]:
train_data_named = WideAndDeepDataset(
    X_wide=X_train[:, :5], X_deep=X_train[:, 2:], y=y_train)
train_loader_named = DataLoader(train_data_named, batch_size=32, shuffle=True)

valid_data_named = WideAndDeepDataset(
    X_wide=X_valid[:, :5], X_deep=X_valid[:, 2:], y=y_valid)
valid_loader_named = DataLoader(valid_data_named, batch_size=32)

test_data_named = WideAndDeepDataset(
    X_wide=X_test[:, :5], X_deep=X_test[:, 2:], y=y_test)
test_loader_named = DataLoader(test_data_named, batch_size=32)

The training and evaluation loops can now access inputs by name instead of relying on their order.

The `**` operator unpacks the dictionary as named arguments.

`model(**inputs)` is equivalent to:

`model(X_wide=inputs["X_wide"], X_deep=inputs["X_deep"])`

In [353]:
def train_named_inputs(model, optimizer, criterion, train_loader, n_epochs):
    model.train()

    for epoch in range(n_epochs):
        total_loss = 0.

        for inputs, y_batch in train_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            y_batch = y_batch.to(device)
            
            y_pred = model(**inputs)

            loss = criterion(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)
        
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

`model(**inputs)` unpacks the input dictionary as named arguments.

For this model, it is equivalent to:

`model(X_wide=inputs["X_wide"], X_deep=inputs["X_deep"])`

In [356]:
def evaluate_named_inputs(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()

    metrics = []

    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            
            y_batch = y_batch.to(device)
            
            y_pred = model(**inputs)
            
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))

The evaluation loop uses the same named-input pattern

### Building Models with Multiple Outputs

A neural network can produce multiple outputs for multitask learning or regularization.

In this example, an auxiliary output encourages the deep path to make useful predictions on its own.

In this example, the main output uses both the wide and deep paths.

The auxiliary output uses only the deep path, encouraging the deep stack to learn useful features on its own.

In [383]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_wide_features, n_deep_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_deep_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU()
        )
        self.output_layer = nn.Linear(n_wide_features + 40, 1)
        self.aux_output_layer = nn.Linear(40, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        return main_output, aux_output

The model returns two predictions:

- `main_output` uses both the wide and deep paths.
- `aux_output` uses only the deep path.

In [386]:
torch.manual_seed(42)

model = WideAndDeepV4(
    n_wide_features=5,
    n_deep_features=n_features - 2
).to(device)

learning_rate = 0.002

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

mse = nn.MSELoss()

Each output has its own loss.

The final training loss is a weighted sum of the main and auxiliary losses.

In [389]:
def train_multiple_outputs(model, optimizer, criterion, train_loader, n_epochs):
    model.train()

    for epoch in range(n_epochs):
        total_loss = 0.

        for inputs, y_batch in train_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}
            
            y_batch = y_batch.to(device)

            y_pred, y_pred_aux = model(**inputs)
            
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)

            loss = 0.8 * main_loss + 0.2 * aux_loss
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)

        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [391]:
train_multiple_outputs(
    model,
    optimizer,
    mse,
    train_loader_named,
    n_epochs
)

Epoch 1/20, Loss: 1.7901
Epoch 2/20, Loss: 0.7735
Epoch 3/20, Loss: 0.6958
Epoch 4/20, Loss: 0.6514
Epoch 5/20, Loss: 0.6174
Epoch 6/20, Loss: 0.5904
Epoch 7/20, Loss: 0.5681
Epoch 8/20, Loss: 0.5498
Epoch 9/20, Loss: 0.5341
Epoch 10/20, Loss: 0.5215
Epoch 11/20, Loss: 0.5100
Epoch 12/20, Loss: 0.5004
Epoch 13/20, Loss: 0.4920
Epoch 14/20, Loss: 0.4840
Epoch 15/20, Loss: 0.4777
Epoch 16/20, Loss: 0.4721
Epoch 17/20, Loss: 0.4679
Epoch 18/20, Loss: 0.4632
Epoch 19/20, Loss: 0.4599
Epoch 20/20, Loss: 0.4570


During evaluation, only the main output is used.

The auxiliary output is included only to regularize the deep path during training.

In [394]:
def evaluate_multiple_outputs(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()

    metrics = []

    with torch.no_grad():
        for inputs, y_batch in data_loader:
            inputs = {name: X.to(device) for name, X in inputs.items()}

            y_batch = y_batch.to(device)
            
            y_pred, _ = model(**inputs)
            
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)

    return aggregate_fn(torch.stack(metrics))

## Building an Image Classifier with PyTorch

We will build a classifier for the Fashion MNIST dataset.

Unlike the earlier regression examples, this task predicts one of 10 clothing classes.

### Using TorchVision to Load the Dataset

TorchVision provides datasets, image transforms, and pretrained computer vision models.

Fashion MNIST contains 60,000 training images and 10,000 test images.

In [398]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)

train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

100.0%


Extracting datasets\FashionMNIST\raw\train-images-idx3-ubyte.gz to datasets\FashionMNIST\raw



100.0%


Extracting datasets\FashionMNIST\raw\train-labels-idx1-ubyte.gz to datasets\FashionMNIST\raw



100.0%


Extracting datasets\FashionMNIST\raw\t10k-images-idx3-ubyte.gz to datasets\FashionMNIST\raw



100.0%

Extracting datasets\FashionMNIST\raw\t10k-labels-idx1-ubyte.gz to datasets\FashionMNIST\raw



Fashion MNIST images are loaded as PIL images with pixel values from 0 to 255.

The transform converts them to float32 tensors and scales pixel values to the 0 to 1 range.

Then, the original training set is split into training and validation sets.

In [404]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

Each image is stored as a tensor with shape:

`[channels, height, width]`

Fashion MNIST images are grayscale, so they have one channel.

In [407]:
X_sample, y_sample = train_data[0]

print("Image shape:", X_sample.shape)
print("Image data type:", X_sample.dtype)

Image shape: torch.Size([1, 28, 28])
Image data type: torch.float32


In [411]:
print("Class name:")
print(train_and_valid_data.classes[y_sample])

Class name:
Ankle boot


### Building the Classifier

We build a custom classification MLP with two hidden layers for Fashion MNIST.

In [436]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)

In [438]:
torch.manual_seed(42)

model = ImageClassifier(n_inputs=28*28, n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)

xentropy = nn.CrossEntropyLoss()

The model is a simple sequence of layers, so it uses `nn.Sequential`.

`nn.Flatten()` reshapes each image from `[1, 28, 28]` to 784 values for the linear layers.

The output layer has 10 neurons, one for each class. No activation function is used after the output layer because `nn.CrossEntropyLoss()` expects logits.

`nn.CrossEntropyLoss()` accepts integer class indices such as `0` to `9` as targets.

It computes cross-entropy directly from logits, which is more efficient and numerically stable than applying `Softmax` before the loss calculation. Apply softmax manually only when class probabilities are needed.

For binary classification, use one output neuron with `nn.BCEWithLogitsLoss()`.

For multilabel binary classification, use one output neuron per label with `nn.BCEWithLogitsLoss()`.

In [443]:
learning_rate = 0.01

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

train(model, optimizer, xentropy, train_loader, n_epochs)

Epoch 1/20, Loss: 1.1357
Epoch 2/20, Loss: 0.6023
Epoch 3/20, Loss: 0.5140
Epoch 4/20, Loss: 0.4736
Epoch 5/20, Loss: 0.4502
Epoch 6/20, Loss: 0.4308
Epoch 7/20, Loss: 0.4149
Epoch 8/20, Loss: 0.4017
Epoch 9/20, Loss: 0.3878
Epoch 10/20, Loss: 0.3775
Epoch 11/20, Loss: 0.3676
Epoch 12/20, Loss: 0.3581
Epoch 13/20, Loss: 0.3499
Epoch 14/20, Loss: 0.3426
Epoch 15/20, Loss: 0.3353
Epoch 16/20, Loss: 0.3287
Epoch 17/20, Loss: 0.3222
Epoch 18/20, Loss: 0.3146
Epoch 19/20, Loss: 0.3096
Epoch 20/20, Loss: 0.3037


To evaluate the classifier, use accuracy.

In [445]:
import torchmetrics

accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [451]:
model.eval()

X_new, y_new = next(iter(valid_loader))

X_new = X_new[:3].to(device)

with torch.no_grad():
    y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)

print(y_pred)

tensor([7, 4, 2], device='cuda:0')


In [453]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Coat', 'Pullover']

For each image, the predicted class is the index of the largest logit.

In [456]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)

print(y_proba.round(decimals=3))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0130, 0.0000, 0.9450, 0.0040,
         0.0380],
        [0.0000, 0.0000, 0.0050, 0.0000, 0.9940, 0.0000, 0.0010, 0.0000, 0.0000,
         0.0000],
        [0.0020, 0.0000, 0.6490, 0.0010, 0.2610, 0.0000, 0.0770, 0.0000, 0.0110,
         0.0000]], device='cuda:0')


The model outputs logits during training. Apply `F.softmax()` when estimated class probabilities are needed.

Label smoothing can be enabled with the `label_smoothing` hyperparameter of `nn.CrossEntropyLoss()`, such as `label_smoothing=0.05`.

In [460]:
y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)

y_top4_probas = F.softmax(y_top4_logits, dim=1)

print(y_top4_probas.round(decimals=3))
print(y_top4_indices)

tensor([[0.9450, 0.0380, 0.0130, 0.0040],
        [0.9940, 0.0050, 0.0010, 0.0000],
        [0.6510, 0.2610, 0.0780, 0.0110]], device='cuda:0')
tensor([[7, 9, 5, 8],
        [4, 2, 6, 8],
        [2, 4, 6, 8]], device='cuda:0')


`torch.topk()` returns the largest logits and their class indices.

For imbalanced classification, use the `weight` argument of `nn.CrossEntropyLoss()` to assign larger weights to rare classes.

## Fine-Tuning Neural Network Hyperparameters with Optuna

Optuna can automatically search for hyperparameter values that improve validation performance.

In this example, we tune the learning rate and the number of neurons in both hidden layers.

In [473]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, 
                            n_hidden2=n_hidden, n_classes=10).to(device)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    train(model, optimizer, xentropy, train_loader, n_epochs=3)

    accuracy.reset()

    model.eval()

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred_logits = model(X_batch)
            
            accuracy.update(y_pred_logits, y_batch)

    validation_accuracy = accuracy.compute()

    return validation_accuracy.item()

`trial.suggest_float()` and `trial.suggest_int()` ask Optuna to select hyperparameter values.

Using `log=True` for `learning_rate` lets Optuna explore different scales more effectively.

In [476]:
torch.manual_seed(42)

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(objective, n_trials=2)

[I 2026-07-06 10:31:27,555] A new study created in memory with name: no-name-97d51697-8541-4e34-9ddd-f5761928f1b5


Epoch 1/3, Loss: 2.2769
Epoch 2/3, Loss: 2.2093
Epoch 3/3, Loss: 2.1164


[I 2026-07-06 10:31:53,142] Trial 0 finished with value: 0.4514000117778778 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.4514000117778778.


Epoch 1/3, Loss: 1.1595
Epoch 2/3, Loss: 0.6154
Epoch 3/3, Loss: 0.5203


[I 2026-07-06 10:32:18,065] Trial 1 finished with value: 0.8100000023841858 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8100000023841858.


Optuna uses validation accuracy as the objective, so the study maximizes the returned score.

The TPE sampler gradually focuses on more promising regions of the hyperparameter space.

In [485]:
print("Best hyperparameters:")
print(study.best_params)

print("\nBest validation accuracy:")
print(study.best_value)

Best hyperparameters:
{'learning_rate': 0.008471801418819975, 'n_hidden': 188}

Best validation accuracy:
0.8100000023841858


More trials can improve the search result, but each trial trains a new model and increases computation time.

Additional hyperparameters such as batch size, optimizer type, number of layers, or activation function can also be tuned, but expanding the search space increases the search cost.

## Saving and Loading PyTorch Models

PyTorch models can be saved either as full model objects or as model weights.

Saving only the model's `state_dict()` is generally safer and more portable.

### Saving and Loading the Full Model

PyTorch can save and load an entire model object using `torch.save(model, path)` and `torch.load(path, weights_only=False)`.

However, this approach relies on Python pickle serialization. It can be unsafe when loading untrusted files and may break if the custom model class or Python environment changes.

For this reason, saving only the model's `state_dict()` is generally recommended.

### Saving and Loading Model Weights

A state dictionary contains the model parameters and registered buffers.

In [492]:
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")

To load saved weights, first recreate the exact same model architecture.

In [501]:
new_model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100,
                            n_classes=10).to(device)

loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)

new_model.load_state_dict(loaded_weights)
new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

In [503]:
X_new = X_new.to(device)

with torch.no_grad():
    y_pred_logits = new_model(X_new)

print(y_pred_logits.argmax(dim=1))

tensor([7, 4, 2], device='cuda:0')


### Saving Model Weights and Hyperparameters

Saving hyperparameters with the state dictionary makes it easier to recreate the correct model architecture later.

In [508]:
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {"n_inputs": 1 * 28 * 28, 
                              "n_hidden1": 300,
                              "n_hidden2": 100,
                              "n_classes": 10
                             }
}

torch.save(model_data, "my_fashion_mnist_model.pt")

In [512]:
loaded_data = torch.load("my_fashion_mnist_model.pt", weights_only=True)

new_model = ImageClassifier(**loaded_data["model_hyperparameters"])

new_model.load_state_dict(loaded_data["model_state_dict"])

new_model.eval()

ImageClassifier(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=300, bias=True)
    (2): ReLU()
    (3): Linear(in_features=300, out_features=100, bias=True)
    (4): ReLU()
    (5): Linear(in_features=100, out_features=10, bias=True)
  )
)

To resume training later, also save the optimizer's `state_dict()`, training epoch, and any training history that is needed.